# Where foundation money goes: grant dollars by recipient location

**Milestone 3 · enrichment view mockup.**

Private-foundation grants are usually seen at the foundation's address. This notebook places each grant from a
Massachusetts-based foundation at the **recipient's** location instead, using the recipient EINs from EIN matching
(and, optionally, fuzzy matching).

| Step | What it does |
|---|---|
| 1 | Load the EIN matching output and keep grants from MA-based foundations |
| 2 | (Optional) add fuzzy-matched rows |
| 3 | Give every matched grant a recipient location: the IRS-registered ZIP, else the ZIP on the grant record |
| 4 | Check the largest grants (a few huge grants can dominate dollar totals) |
| 5 | Summary numbers: how much stays in MA, top recipient states and places |
| 6 | Map grant dollars by recipient ZIP code area (ZCTA) inside Massachusetts |
| 7 | Save the tables and the map |

Run top to bottom. Edit only the **Config** cell.

## Config

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Colab: True = upload the input files from your computer and download the results as a zip.
#        False = read from / save to Google Drive.
UPLOAD_FROM_COMPUTER = True

# Folder that holds the EIN matching outputs (pf_recipients_matched.parquet, bmf_clean.parquet).
# If a file is not found there, the Setup cell searches all of MyDrive for it by name.
OUTPUTS_DIR = Path("/content/drive/MyDrive/pf_recipient_matching") if IN_COLAB else Path("data")
MATCHED_FILE = "pf_recipients_matched.parquet"
BMF_FILE = "bmf_clean.parquet"

# Optional: fuzzy matching output (written by src/fuzzy_matching/fuzzy_match.py --sample filer_ma).
# Leave as None to map EIN matching results only.
FUZZY_FILE = "fuzzy_matches_filer_ma.csv"   # set to None to map EIN matching results only
EXPECTED_FUZZY_ROWS = 469         # grant rows fuzzy matching added for MA-based foundations (its README)

FOUNDATION_STATE = "MA"
TOP_N = 15
MAP_DIR = Path("/content/zcta_cache") if IN_COLAB else Path("zcta_cache")
SAVE_DIR_NAME = "grant_recipient_map"   # created next to the outputs folder

# Census cartographic boundary files (2020). Downloaded once and cached.
ZCTA_URL = "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_zcta520_500k.zip"
STATE_URL = "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_state_500k.zip"
MA_ZIP3 = {f"{i:03d}" for i in range(10, 28)} | {"055"}

## Setup

In [ ]:
import subprocess

if IN_COLAB:
    if UPLOAD_FROM_COMPUTER:
        from google.colab import files
        Path("/content/upload").mkdir(exist_ok=True)
        print(f"Choose these files from your computer (you can select several at once):\n"
              f"  {MATCHED_FILE}, {FUZZY_FILE or '(no fuzzy file)'}, and optionally {BMF_FILE}")
        for fname, data in files.upload().items():
            (Path("/content/upload") / fname).write_bytes(data)
        OUTPUTS_DIR = Path("/content/upload")
    else:
        from google.colab import drive
        drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "geopandas", "pyarrow", "folium", "mapclassify"],
                   check=True)

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def locate(name: str, required: bool = True):
    p = OUTPUTS_DIR / name
    if p.exists():
        return p
    roots = [OUTPUTS_DIR] + ([Path("/content/drive/MyDrive")] if IN_COLAB and not UPLOAD_FROM_COMPUTER else [])
    for root in roots:
        if root.exists():
            print(f"  searching {root} for {name} ...")
            hit = next(root.rglob(name), None)
            if hit:
                return hit
    if required:
        raise FileNotFoundError(f"Could not find {name}. Set OUTPUTS_DIR in the Config cell (currently {OUTPUTS_DIR}).")
    return None


MATCHED_PATH = locate(MATCHED_FILE)
BMF_PATH = locate(BMF_FILE, required=False)
FUZZY_PATH = locate(FUZZY_FILE, required=True) if FUZZY_FILE else None
if IN_COLAB and UPLOAD_FROM_COMPUTER:
    SAVE_DIR = Path("/content") / SAVE_DIR_NAME
else:
    SAVE_DIR = MATCHED_PATH.parent.parent / SAVE_DIR_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)
for k, v in {"matched": MATCHED_PATH, "bmf": BMF_PATH, "fuzzy": FUZZY_PATH}.items():
    print(f"{k:8s} {v}")
print("Outputs go to", SAVE_DIR)


def zip5(s: pd.Series) -> pd.Series:
    d = s.fillna("").astype(str).str.replace(r"\D", "", regex=True)
    d = d.where(~d.str.len().isin([4, 8]), d.str.zfill(9).where(d.str.len() == 8, d.str.zfill(5)))
    return d.str[:5].where(d.str.len() >= 5, "")


def money(x: float) -> str:
    return f"${x / 1e9:,.2f}B" if abs(x) >= 1e9 else f"${x / 1e6:,.1f}M"

## 1. Load grants from MA-based foundations

`filer_state` is the foundation's state (from the 990-PF header). Rows set aside during EIN matching
(individuals, foreign recipients, "SEE ATTACHED") are dropped, so the numbers line up with the match-rate table.

In [ ]:
COLS = ["filer_ein", "taxyear", "line_number", "filer_state", "amount", "recipient_class", "match_method",
        "is_matched", "matched_ein", "rcp_name_full", "rcp_name_norm", "rcp_city", "rcp_state_n", "rcp_zip5",
        "matched_city", "matched_state", "matched_zip5", "matched_addr_source"]
m = pd.read_parquet(MATCHED_PATH, columns=COLS)
g = m[(m.filer_state == FOUNDATION_STATE) & (m.match_method != "set_aside")].copy()
g["amount"] = pd.to_numeric(g.amount, errors="coerce").fillna(0)
g["source"] = np.where(g.is_matched, "ein_matching", "")
print(f"Grants from {FOUNDATION_STATE}-based foundations (eligible): {len(g):,}  "
      f"matched by EIN matching: {g.is_matched.sum():,} ({g.is_matched.mean():.1%})")

## 2. (Optional) add fuzzy-matched grants

The fuzzy output has **one row per unique recipient** (normalized name + state + ZIP), not one row per grant.
It is joined back to grants the same way `match_rate_table.py` does: on `rcp_name_norm`, `rcp_state_n` and
`rcp_zip5`, only for grants EIN matching left `unmatched` or `ambiguous`. Grants marked `gov_name_review` stay
unmatched, as in the final match-rate table. The matched organization's ZIP comes from the fuzzy file
(`matched_zip5`); its city comes from the IRS Business Master File when the EIN is there.

In [ ]:
if FUZZY_PATH:
    fz = pd.read_csv(FUZZY_PATH, dtype=str, keep_default_na=False)
    fz = fz[fz.match_status == "MATCHED"].copy()
    for c in ["name_norm", "state", "zip5", "matched_ein", "matched_zip5"]:
        fz[c] = fz[c].astype(str).str.strip()
    fz["matched_ein"] = fz.matched_ein.str.zfill(9)
    fz = fz.drop_duplicates(["name_norm", "state", "zip5"]).set_index(["name_norm", "state", "zip5"])
    s = lambda c: g[c].fillna("").astype(str).str.strip()
    keys = pd.MultiIndex.from_arrays([s("rcp_name_norm"), s("rcp_state_n"), s("rcp_zip5")])
    to_fuzzy = g.match_method.isin(["unmatched", "ambiguous"]).values
    hit = to_fuzzy & keys.isin(fz.index)
    rows = fz.reindex(keys[hit])
    g.loc[hit, "matched_ein"] = rows.matched_ein.values
    g.loc[hit, "matched_zip5"] = rows.matched_zip5.values
    g.loc[hit, "matched_state"] = keys[hit].get_level_values(1)   # fuzzy matching stays within the state
    g.loc[hit, "matched_city"] = ""
    g.loc[hit, "matched_addr_source"] = "fuzzy"
    g.loc[hit, "is_matched"] = True
    g.loc[hit, "source"] = "fuzzy_matching"
    if BMF_PATH:
        bmf = pd.read_parquet(BMF_PATH, columns=["EIN", "CITY"]).drop_duplicates("EIN").set_index("EIN")
        g.loc[hit, "matched_city"] = g.loc[hit, "matched_ein"].map(bmf.CITY).fillna("").values
    print(f"Fuzzy-matched recipients in the file: {len(fz):,}; grants they cover: {hit.sum():,} "
          f"(fuzzy README: {EXPECTED_FUZZY_ROWS:,})")
    if FOUNDATION_STATE == "MA" and hit.sum() != EXPECTED_FUZZY_ROWS:
        print("  NOTE: differs from the fuzzy README. Check that the file is fuzzy_matches_filer_ma.csv.")
print(f"Matched now: {g.is_matched.sum():,} of {len(g):,} ({g.is_matched.mean():.1%})")
print(g.loc[g.is_matched, "source"].value_counts().to_string())

## 3. Recipient location for each matched grant

Preferred: the ZIP registered with the IRS for the matched EIN (`matched_zip5`). If that is missing (Pub 78 gives
no ZIP, and government EINs are not in the BMF), the ZIP written on the grant record is used.

In [ ]:
mg = g[g.is_matched].copy()
clean = lambda s: s.fillna("").astype(str).str.upper().str.strip()
mz, rz = zip5(mg.matched_zip5), zip5(mg.rcp_zip5)
use_irs = mz != ""
# ZIP, city and state always come from the same address, so a place is never stitched from two records
mg["loc_source"] = np.where(use_irs, "irs_registered", np.where(rz != "", "grant_record", "none"))
mg["loc_zip5"] = np.where(use_irs, mz, rz)
mg["loc_city"] = np.where(use_irs, clean(mg.matched_city), clean(mg.rcp_city))
mg["loc_state"] = np.where(use_irs, clean(mg.matched_state), clean(mg.rcp_state_n))
# no ZIP at all: fall back to the IRS-registered state/city (Pub 78 has these without a ZIP)
nozip = mg.loc_source == "none"
mg.loc[nozip, "loc_state"] = np.where(clean(mg.matched_state[nozip]) != "", clean(mg.matched_state[nozip]),
                                      clean(mg.rcp_state_n[nozip]))
# the ZIP decides the state when they disagree or the state is missing (MA ZIP3 = 010-027, 055)
zip_ma = mg.loc_zip5.str[:3].isin(MA_ZIP3)
mg.loc[zip_ma, "loc_state"] = "MA"
mg.loc[(mg.loc_zip5 != "") & ~zip_ma & (mg.loc_state == "MA"), "loc_state"] = ""

print(mg.loc_source.value_counts().to_string())
print(f"\nMatched grants with no recipient ZIP: {(mg.loc_zip5 == '').sum():,} "
      f"({money(mg.amount[mg.loc_zip5 == ''].sum())})")

## 4. Check the largest grants

Dollar totals can be dominated by a handful of very large grants. Look at these before reading the map.

In [ ]:
total = mg.amount.sum()
top = mg.nlargest(TOP_N, "amount")[["rcp_name_full", "loc_city", "loc_state", "amount", "source"]]
top["share_of_total"] = (top.amount / total * 100).round(1)
print(f"Total matched grant dollars: {money(total)} across {len(mg):,} grants")
print(f"Top 10 grants = {mg.nlargest(10, 'amount').amount.sum() / total:.1%} of all dollars; "
      f"median grant = ${mg.amount.median():,.0f}")
top

## 5. Where the money goes: summary numbers

In [ ]:
in_ma = mg.loc_state == "MA"
known = mg.loc_state != ""
summary = pd.DataFrame([
    {"metric": "Matched grants", "value": f"{len(mg):,}"},
    {"metric": "Matched grant dollars", "value": money(total)},
    {"metric": "Share of dollars to recipients in MA", "value": f"{mg.amount[in_ma].sum() / mg.amount[known].sum():.1%}"},
    {"metric": "Share of grants to recipients in MA", "value": f"{in_ma.sum() / known.sum():.1%}"},
    {"metric": "Recipient state unknown (dollars)", "value": money(mg.amount[~known].sum())},
])
print(summary.to_string(index=False))

by_state = (mg[known].groupby("loc_state")
            .agg(grants=("amount", "size"), dollars=("amount", "sum")).sort_values("dollars", ascending=False))
by_state["share_dollars_%"] = (100 * by_state.dollars / by_state.dollars.sum()).round(1)
print("\nTop recipient states:")
print(by_state.head(10).to_string())

ma = mg[in_ma & (mg.loc_zip5 != "")]
by_zip = (ma.groupby("loc_zip5")
          .agg(grants=("amount", "size"), dollars=("amount", "sum"), recipients=("matched_ein", "nunique"),
               city=("loc_city", lambda s: s.mode().iat[0] if len(s.mode()) else ""))
          .sort_values("dollars", ascending=False))
by_zip["share_dollars_%"] = (100 * by_zip.dollars / by_zip.dollars.sum()).round(1)
print(f"\nMA recipient ZIPs: {len(by_zip):,}. Top {TOP_N}:")
print(by_zip.head(TOP_N).to_string())

by_city = (ma.groupby("loc_city").agg(grants=("amount", "size"), dollars=("amount", "sum"))
           .sort_values("dollars", ascending=False))
by_city["share_dollars_%"] = (100 * by_city.dollars / by_city.dollars.sum()).round(1)
print(f"\nTop {TOP_N} MA recipient cities:")
print(by_city.head(TOP_N).to_string())

## 6. Map: grant dollars by recipient ZIP code area (Massachusetts)

ZIP codes are drawn as Census ZIP Code Tabulation Areas (ZCTAs). PO-box-only ZIPs have no ZCTA, so their dollars
are reported but not drawn. Colors use a log scale, because dollars range from hundreds to millions.

In [ ]:
import io, zipfile
import requests
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

MAP_DIR.mkdir(parents=True, exist_ok=True)


def fetch_shapes(url: str) -> gpd.GeoDataFrame:
    folder = MAP_DIR / Path(url).stem
    if not folder.exists():
        print(f"  downloading {Path(url).name} ...")
        r = requests.get(url, timeout=600)
        r.raise_for_status()
        zipfile.ZipFile(io.BytesIO(r.content)).extractall(folder)
    return gpd.read_file(next(folder.glob("*.shp")))


zcta = fetch_shapes(ZCTA_URL)
zcol = [c for c in zcta.columns if c.upper().startswith("ZCTA5")][0]
zcta = zcta[zcta[zcol].str[:3].isin(MA_ZIP3)].rename(columns={zcol: "zcta"})
state = fetch_shapes(STATE_URL)
ma_outline = state[state.STATEFP == "25"]
print(f"MA ZCTAs: {len(zcta):,}")

gmap = zcta.merge(by_zip[["grants", "dollars", "recipients", "city"]], left_on="zcta", right_index=True, how="left")
drawn = by_zip.index.isin(zcta.zcta)
print(f"Recipient ZIPs drawn on the map: {drawn.sum():,} of {len(by_zip):,} "
      f"({by_zip.dollars[drawn].sum() / by_zip.dollars.sum():.1%} of MA dollars); "
      f"not drawn (PO-box-only or retired ZIPs): {money(by_zip.dollars[~drawn].sum())}")

fig, ax = plt.subplots(figsize=(16, 10))
gmap[gmap.dollars.isna()].plot(ax=ax, color="#EDEDED", edgecolor="#FFFFFF", linewidth=0.2)
has = gmap[gmap.dollars > 0]
has.plot(ax=ax, column="dollars", cmap="YlGnBu", edgecolor="#FFFFFF", linewidth=0.2,
         norm=LogNorm(vmin=max(has.dollars.min(), 1), vmax=has.dollars.max()), legend=True,
         legend_kwds={"label": "Grant dollars received (log scale)", "shrink": 0.6})
ma_outline.boundary.plot(ax=ax, color="#555555", linewidth=0.6)
ax.set_axis_off()
ax.set_title(f"Where grants from Massachusetts-based foundations go\n"
             f"Grant dollars by recipient ZIP code, {money(by_zip.dollars.sum())} to MA recipients", fontsize=16)
fig.text(0.12, 0.06, "Gray: no matched grants. Recipient location = IRS-registered address of the matched EIN, "
         "else the address on the grant record.", fontsize=10, color="#555555")
png_path = SAVE_DIR / "grant_dollars_by_recipient_zip_MA.png"
fig.savefig(png_path, dpi=200, bbox_inches="tight")
plt.show()

### Interactive version (hover a ZIP to see its numbers)

In [ ]:
try:
    import folium
    fmap = has.to_crs(4326).copy()
    fmap["dollars_fmt"] = fmap.dollars.map(lambda v: f"${v:,.0f}")
    fmap["log_dollars"] = np.log10(fmap.dollars)
    mobj = folium.Map(location=[42.25, -71.8], zoom_start=8, tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
                      attr="Tiles &copy; Esri &mdash; Esri, HERE, Garmin, OpenStreetMap contributors", max_zoom=16)
    folium.Choropleth(geo_data=fmap.__geo_interface__, data=fmap, columns=["zcta", "log_dollars"],
                      key_on="feature.properties.zcta", fill_color="YlGnBu", fill_opacity=0.75, line_opacity=0.2,
                      legend_name="Grant dollars received (log10)").add_to(mobj)
    folium.GeoJson(fmap[["zcta", "city", "grants", "recipients", "dollars_fmt", "geometry"]].__geo_interface__,
                   style_function=lambda f: {"fillOpacity": 0, "weight": 0},
                   tooltip=folium.GeoJsonTooltip(fields=["zcta", "city", "grants", "recipients", "dollars_fmt"],
                                                 aliases=["ZIP", "City", "Grants", "Recipients", "Dollars"])
                   ).add_to(mobj)
    html_path = SAVE_DIR / "grant_dollars_by_recipient_zip_MA.html"
    mobj.save(str(html_path))
    display(mobj)
except ImportError:
    print("folium not installed; skipping the interactive map")

## 7. Save

In [ ]:
summary.to_csv(SAVE_DIR / "summary.csv", index=False)
by_state.to_csv(SAVE_DIR / "dollars_by_recipient_state.csv")
by_zip.to_csv(SAVE_DIR / "dollars_by_recipient_zip_MA.csv")
by_city.to_csv(SAVE_DIR / "dollars_by_recipient_city_MA.csv")
top.to_csv(SAVE_DIR / "largest_grants.csv", index=False)
for p in sorted(SAVE_DIR.iterdir()):
    print(f"{p.name:45s} {p.stat().st_size / 1e3:8.1f} KB")

# Download everything to your computer as one zip
if IN_COLAB and UPLOAD_FROM_COMPUTER:
    import shutil
    from google.colab import files
    files.download(shutil.make_archive(f"/content/{SAVE_DIR_NAME}", "zip", SAVE_DIR))